# AI-Based Resume Screening System

## Objective
To rank resumes against a job description using semantic similarity
and required-skill coverage.

## Methodology
1. Load the job description and resumes.
2. Extract and preprocess text.
3. Generate embeddings using Sentence Transformers.
4. Calculate cosine similarity.
5. Calculate skill coverage.
6. Combine scores and rank candidates.
7. Export the ranking as CSV and PNG.

Final Score = 60% Semantic Similarity + 40% Skill Coverage

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from screen import (
    BASE_DIR,
    RESUME_DIR,
    JOB_DESCRIPTION_PATH,
    RESULTS_DIR,
    extract_text,
    preprocess_text,
    find_skills,
    extract_years_experience,
    create_embeddings,
    min_max_scale,
    REQUIRED_SKILLS,
    SEMANTIC_WEIGHT,
    SKILL_WEIGHT,
)

from sklearn.metrics.pairwise import cosine_similarity

print("Libraries imported successfully!")

In [ ]:
job_description = extract_text(JOB_DESCRIPTION_PATH)

resume_files = sorted(
    file for file in RESUME_DIR.iterdir()
    if file.suffix.lower() in {".txt", ".pdf", ".docx"}
)

print("Job description loaded:", bool(job_description.strip()))
print("Number of resumes:", len(resume_files))

for file in resume_files:
    print("-", file.name)

In [ ]:
cleaned_job_description = preprocess_text(job_description)

resumes = []

for file in resume_files:
    original_text = extract_text(file)

    if not original_text.strip():
        continue

    resumes.append({
        "resume_file": file.name,
        "original_text": original_text,
        "cleaned_text": preprocess_text(original_text),
        "matched_skills": find_skills(original_text),
        "missing_skills": [
            skill for skill in REQUIRED_SKILLS
            if skill not in find_skills(original_text)
        ],
        "years_experience": extract_years_experience(original_text),
        "skill_coverage": (
            len(find_skills(original_text)) / len(REQUIRED_SKILLS)
        ),
    })

print("Readable resumes:", len(resumes))
print("\nSample preprocessed text:")
print(resumes[0]["cleaned_text"][:300])

In [ ]:
documents = [cleaned_job_description] + [
    resume["cleaned_text"] for resume in resumes
]

embeddings, embedding_method = create_embeddings(documents)

print("Embedding method:", embedding_method)
print("Number of documents:", len(documents))

In [ ]:
similarities = cosine_similarity(
    embeddings[0:1],
    embeddings[1:]
)[0]

for resume, similarity in zip(resumes, similarities):
    resume["semantic_similarity"] = float(similarity)

df = pd.DataFrame(resumes)

df["semantic_scaled"] = min_max_scale(
    df["semantic_similarity"]
)

df["final_score"] = (
    SEMANTIC_WEIGHT * df["semantic_scaled"]
    + SKILL_WEIGHT * df["skill_coverage"]
) * 100

df = df.sort_values(
    "final_score", ascending=False
).reset_index(drop=True)

df.insert(0, "rank", range(1, len(df) + 1))

df[[
    "rank",
    "resume_file",
    "semantic_similarity",
    "skill_coverage",
    "final_score",
    "years_experience",
]]

In [ ]:
for _, row in df.iterrows():
    print(f"\nRank {row['rank']}: {row['resume_file']}")
    print(f"Final score: {row['final_score']:.2f}/100")
    print("Matched skills:", ", ".join(row["matched_skills"]))
    print("Missing skills:", ", ".join(row["missing_skills"]))

In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

output_columns = [
    "rank",
    "resume_file",
    "semantic_similarity",
    "skill_coverage",
    "final_score",
    "matched_skills",
    "missing_skills",
    "years_experience",
]

report = df[output_columns].copy()
report["matched_skills"] = report["matched_skills"].apply(
    lambda skills: ", ".join(skills)
)
report["missing_skills"] = report["missing_skills"].apply(
    lambda skills: ", ".join(skills)
)

report.to_csv(RESULTS_DIR / "ranking.csv", index=False)

chart_df = df.sort_values("final_score", ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(chart_df["resume_file"], chart_df["final_score"])
plt.xlabel("Final Score (0–100)")
plt.ylabel("Resume")
plt.title("AI-Based Resume Screening — Candidate Ranking")
plt.xlim(0, 100)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "ranking.png", dpi=200)
plt.show()

print("CSV and chart saved successfully!")